# Predict


In [ ]:
import geopandas as gpd
import rasterio as rio
import numpy as np

import xdem
import geoutils as gu
import subkart
import importlib
import joblib


In [ ]:
dem = xdem.DEM(rio.open("../DEM25Norge_west_norway.tif"))
# Output shape (rows, cols) from DEM
out_shape = dem.data.shape[-2:]  # loads DEM if not loaded
transform = dem.transform


In [ ]:
marine_vanntyper = gpd.read_parquet("gs://niva-geodata/MarintNaturKart/mdir/NyTypologi2022.geo.parquet").to_crs(dem.crs)

## Predict for AOI

In [ ]:
gdf_bunn_moere = gpd.read_file("https://storage.googleapis.com/niva-geodata/MarintNaturKart/bunn_in_kommuner_sea.geojson")
gdf_clip = gdf_bunn_moere.to_crs(dem.crs)

vec_clip = gu.Vector(gdf_clip)

In [ ]:
vect_rasterized = vec_clip.create_mask(dem)

In [ ]:

arr2d = np.array(dem.data, dtype=np.float32)
arr2d[~vect_rasterized.data] = np.nan
clipped_dem = xdem.DEM.from_array(arr2d, transform=dem.transform, crs=dem.crs)

In [ ]:
clipped_dem.plot(cmap="terrain", title="Clipped DEM over Møre og Romsdal")

In [ ]:

clipped_marine_vanntyper = gdf_clip.overlay(marine_vanntyper)

In [ ]:


X, valid_attrs = subkart.features.build(clipped_dem, clipped_marine_vanntyper)

In [ ]:
classifier = joblib.load("../data_generated/classifier.joblib")


In [ ]:
Y_pred = classifier.predict(X[valid_attrs])

In [ ]:
pred_map = np.full(clipped_dem.data.shape[-2:], np.nan, dtype=np.float32)
pred_map[valid_attrs] = Y_pred.astype(np.float32)

In [ ]:

pred_raster = gu.Raster.from_array(pred_map, transform=clipped_dem.transform, crs=clipped_dem.crs)

# Vectorize output

In [ ]:
pred_vec = pred_raster.polygonize()

In [ ]:
# Map raster_value (0/1) to BunnType using existing mapping_values
reverse_map = {v: k for k, v in subkart.labelling.BUNNTYPE_MAPPING.items()}
pred_vec["BunnType"] = pred_vec["raster_value"].map(reverse_map)

pred_vec["BunnType"]

In [ ]:
gdf_final = pred_vec.ds.dissolve(by="BunnType", as_index=False, method="coverage")

In [ ]:
gdf_final = gdf_final.to_crs("EPSG:25833")

In [ ]:
fname = subkart.utils.to_filename(f"nisjedata-substrat-{classifier.__class__.__name__.lower()}", "moere-og-romsdal", "latest", gdf_final.crs.to_epsg())
gdf_final = gdf_final.drop(columns=["depth_range", "class"], errors="ignore")
gdf_final.to_file(f"{fname}.geojson", driver="GeoJSON")

In [ ]:
gdf_final.to_file(f"{fname}.gpkg", layer="soft_hard_bottom", driver="GPKG")